# 02 · DPO 訓練：用陷阱題練習本訓練模型

**做法（方案 A）**：4-bit Qwen 底模 + 你的 SFT adapter，**繼續訓練同一個 adapter**。TRL 會自動複製一份 SFT adapter 當作「參考模型」，固定不動。

```
Qwen3-4B（4-bit，不訓練）
 ├─ default adapter ← 從 SFT adapter 出發，DPO 訓練這個
 └─ ref adapter     ← SFT adapter 的複本，固定不動，當作比較的基準
```

訓練完只存 `default`，得到一個「SFT+DPO」adapter，可以跟原本的 SFT adapter 互相替換。

**Kaggle 設定**
- Accelerator：**GPU T4 x2**（只會用到第一張，原因見第 2 格）
- Internet：**On**
- Secrets：`HF_TOKEN`（**write** 權限，最後要上傳 adapter）

**執行方式**：Save Version → **Save & Run All (Commit)**
**預估時間**：約 2～3 小時（大部分是訓練）。中途會自動存 checkpoint。

## 1. 安裝套件（鎖定版本）

這些版本跟 Claude 在本機用小模型**實際測試過整個 DPO 流程**的版本一樣。

| 套件 | 版本 | 用途 |
|---|---|---|
| `trl` | 1.18.2 | `DPOTrainer`。這一版會自動把 SFT adapter 複製成參考 adapter |
| `peft` | 0.22.0 | 載入 LoRA adapter。0.20 以後才能穩定複製 adapter |
| `transformers` | 5.24.0 | 載入 Qwen 模型 |
| `bitsandbytes` | 最新 | 4-bit 量化（QLoRA） |
| `accelerate`、`datasets` | 1.16.0、5.2.0 | 訓練工具、讀資料 |
| `trackio` | 0.45.1 | 訓練曲線儀表板（你 10 月練 SFT 時用過的那個） |

這次**不用裝 vLLM**，訓練只用 transformers。

In [ ]:
!pip install -q -U "trl==1.18.2" "peft==0.22.0" "transformers==5.24.0" "accelerate==1.16.0" "datasets==5.2.0" "trackio==0.45.1" bitsandbytes

## 2. 只用一張 GPU、記錄版本、登入 HF

**`CUDA_VISIBLE_DEVICES=0`**：Kaggle 給兩張 T4，但 4-bit 模型放在兩張卡上用 DataParallel 跑很容易出錯。4B 模型一張 T4 就放得下，所以只讓程式看到第一張。**一定要在 `import torch` 之前設定**，之後再設就沒用了。

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib.metadata as md_, subprocess, json, time
VERSIONS = {p: md_.version(p) for p in ["trl", "peft", "transformers", "accelerate", "datasets", "bitsandbytes", "torch"]}
print(VERSIONS)

import torch
ON_GPU = torch.cuda.is_available()
GPU_INFO = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip() if ON_GPU else "cpu"
print("GPU:", GPU_INFO, "| 程式看得到的 GPU 數:", torch.cuda.device_count())
assert ON_GPU, "沒有 GPU！請到 Settings → Accelerator 選 GPU T4 x2"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
print("HF 使用者:", whoami()["name"])

## 3. 參數

### 模型和資料

| 參數 | 值 | 意思 |
|---|---|---|
| `BASE_ID` | Qwen3-4B-Instruct-2507 | 底模，跟 SFT 時一樣 |
| `ADAPTER_ID` | grid-dispatch-A-instruct-qlora-v1 | 你的 SFT adapter，DPO 從這裡出發 |
| `PAIRS_ID` | grid-dispatch-dpo-pairs | 01b 做的練習本（train）和小考卷（test） |
| `OUT_REPO` | grid-dispatch-A-instruct-qlora-dpo-v1 | 訓練完的 DPO adapter 要上傳到這裡（私人） |

### DPO 的參數（最重要）

| 參數 | 值 | 意思 | 調大會怎樣 / 調小會怎樣 |
|---|---|---|---|
| `BETA` | 0.1 | 「不准離 SFT 太遠」的力道 | 大 → 比較保守、改變少；小 → 改變大，但可能學壞。0.1 是 DPO 論文和 TRL 的預設值 |
| `LOSS_TYPE` | `["sigmoid", "sft"]` | sigmoid = 標準 DPO loss；sft = 同時對正確答案做一點 SFT | 加 sft 是為了防止 02 開頭提過的陷阱：DPO 有時會把正確答案的機率**也一起壓低** |
| `LOSS_WEIGHTS` | `[1.0, 1.0]` | 兩個 loss 的比重 | sft 比重越大，越像 SFT |
| `LEARNING_RATE` | 1e-5 | 每一步調整多少 | TRL 文件建議 adapter 訓練用 1e-5 左右。太大會把 SFT 學到的東西洗掉 |
| `EPOCHS` | 1 | 練習本從頭到尾練幾遍 | 練習本是規則造出來的、比較好分辨，多練容易過擬合，先練 1 遍 |

### 記憶體和速度

| 參數 | 值 | 意思 |
|---|---|---|
| `BATCH_SIZE` | 2 | 每次送進 GPU 的配對數。每筆配對要算 chosen 和 rejected 兩條，所以實際是 4 條句子 |
| `GRAD_ACCUM` | 8 | 累積 8 次再更新一次 → **有效批次 = 2 × 8 = 16 筆配對**。用時間換記憶體 |
| `MAX_LENGTH` | 1536 | prompt + 答案的最大 token 數。設大一點不會多佔記憶體：每一批只補齊到那批最長的句子。第 6 格會用真正的 tokenizer 檢查最長有多長 |
| `EVAL_STEPS` | 25 | 每 25 步考一次小考卷 |
| `SAVE_STEPS` | 50 | 每 50 步存一次 checkpoint，萬一中斷可以接著跑 |

1,692 筆 ÷ 16 ≈ **106 步**。

In [ ]:
BASE_ID    = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_ID = "yoyu001/grid-dispatch-A-instruct-qlora-v1"
PAIRS_ID   = "yoyu001/grid-dispatch-dpo-pairs"
OUT_REPO   = "yoyu001/grid-dispatch-A-instruct-qlora-dpo-v1"
PUSH_TO_HUB = True

BETA          = 0.1
LOSS_TYPE     = ["sigmoid", "sft"]
LOSS_WEIGHTS  = [1.0, 1.0]
LEARNING_RATE = 1e-5
EPOCHS        = 1
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
MAX_LENGTH    = 1536
EVAL_STEPS    = 25
SAVE_STEPS    = 50
SEED          = 42

TRACKIO_PROJECT = "grid-dispatch-dpo"
TRACKIO_SPACE   = None   # None = 只存在本機（建立私人 Space 會被 HF 回 402）
RUN_NAME        = f"dpo-beta{BETA}-lr{LEARNING_RATE}-ep{EPOCHS}"

WORK    = "/kaggle/working"
CKPT    = f"{WORK}/dpo_ckpt"
OUT_DIR = f"{WORK}/dpo_adapter"
LOG_DIR = f"{OUT_DIR}/training_logs"
os.makedirs(LOG_DIR, exist_ok=True)
os.environ["TRACKIO_DIR"] = f"{WORK}/trackio"   # 本機也存一份紀錄，會出現在 Output

## 3.5 Trackio：訓練紀錄

Trackio 會自動記錄 loss、rewards、GPU 使用率等指標。這次只存在**本機**：

- 本來想同步到一個私人 HF Space，讓背景執行時也能即時看曲線，但 HF 回傳 **402 Payment Required**，拒絕建立私人 Space
- 所以改成只存在 `/kaggle/working/trackio/`，跑完後會出現在 Output

| 設定 | 值 | 意思 |
|---|---|---|
| `report_to` | `"trackio"` | 訓練紀錄送給 Trackio |
| `project` | grid-dispatch-dpo | 專案名稱。之後調參數再練一次，同一個專案裡的多次訓練可以疊在一起比較 |
| `run_name` | `dpo-beta0.1-lr1e-05-ep1` | 這次訓練的名稱。**把關鍵參數寫進名字**，比較時一眼就知道是哪一組 |
| `TRACKIO_SPACE` | None | 不同步到 Space。之後有付費方案或想用公開 Space，把這裡改成 `"yoyu001/空間名稱"` 就好 |

**跑完怎麼看**：
- 最快：第 11 格會畫一張 PNG 曲線圖（`dpo_curves.png`），跟 adapter 一起上傳到 HF
- 互動儀表板：把 Output 的 `trackio/` 資料夾下載到電腦，裝好 trackio 之後執行
  `TRACKIO_DIR=下載的資料夾路徑 trackio show --project grid-dispatch-dpo`

In [ ]:
TRACKIO_SPACE_ID = None
if TRACKIO_SPACE:
    from huggingface_hub import HfApi
    try:
        HfApi().create_repo(TRACKIO_SPACE, repo_type="space", space_sdk="gradio", private=True, exist_ok=True)
        TRACKIO_SPACE_ID = TRACKIO_SPACE
        print(f"✅ Trackio 儀表板（私人）：https://huggingface.co/spaces/{TRACKIO_SPACE}")
    except Exception as e:
        print("⚠️ 無法建立 Trackio Space，改成只存在本機（不影響訓練）：", repr(e)[:300])
else:
    print(f"Trackio 紀錄只存在本機：{os.environ['TRACKIO_DIR']}（會出現在 Output 的 trackio/ 資料夾）")

## 4. 讀練習本

01b 存的欄位有 `prompt / chosen / rejected`（TRL 要的），以及 `id / category / source`（我們自己分析用）。訓練時把後三個拿掉，因為 TRL 不認得這些欄位；原始資料另外留一份，後面會用到。

In [ ]:
from datasets import load_dataset
import collections

pairs = load_dataset(PAIRS_ID)
print(pairs)
print("練習本來源:", dict(collections.Counter(pairs["train"]["source"])))

EXTRA = ["id", "category", "source"]
train_ds = pairs["train"].remove_columns(EXTRA)
eval_ds  = pairs["test"].remove_columns(EXTRA)

## 5. 看一下 SFT 時的設定

adapter repo 裡有你 SFT 時存的 `training_logs/run_config.json`。印出來存進紀錄，之後寫報告時可以比較 SFT 和 DPO 的設定。

In [ ]:
from huggingface_hub import hf_hub_download
try:
    SFT_RUN_CONFIG = json.load(open(hf_hub_download(ADAPTER_ID, "training_logs/run_config.json")))
except Exception as e:
    SFT_RUN_CONFIG = {"error": repr(e)}
print(json.dumps(SFT_RUN_CONFIG, ensure_ascii=False, indent=2)[:3000])

## 6. 載入 tokenizer，檢查句子長度

tokenizer 從 **SFT adapter 的 repo** 載入，裡面有 SFT 時用的 chat template，確保 DPO 看到的格式跟 SFT 時一模一樣。

順便算一下最長的「prompt + 答案」有幾個 token。超過 `MAX_LENGTH` 的話會被截斷，模型就看不到完整的答案。

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def n_tokens(ex):
    p = tokenizer.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
    c = max(ex["chosen"][0]["content"], ex["rejected"][0]["content"], key=len)
    return len(tokenizer(p + c)["input_ids"])

lens = [n_tokens(ex) for ex in train_ds]   # 全部檢查，只要幾秒
print(f"{len(lens)} 筆：平均 {sum(lens)/len(lens):.0f}、最長 {max(lens)} tokens（上限 {MAX_LENGTH}）")
assert max(lens) < MAX_LENGTH, "有句子會被截斷，請把 MAX_LENGTH 調大"

## 7. 載入模型：4-bit 底模 + SFT adapter

### 4-bit 量化設定（QLoRA）

| 參數 | 值 | 意思 |
|---|---|---|
| `load_in_4bit` | True | 權重用 4-bit 存。4B 模型從約 8 GB 降到約 2.5 GB |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4，QLoRA 論文提出的 4-bit 格式，專門針對神經網路權重的分布設計 |
| `bnb_4bit_use_double_quant` | True | 連量化用的參數本身也再壓縮一次，再省一點記憶體 |
| `bnb_4bit_compute_dtype` | float16 | 計算時先還原成 fp16 再算。**T4 不支援 bf16**，所以用 fp16 |

### 三個步驟

1. **載入 4-bit 底模**
2. **`prepare_model_for_kbit_training`**：4-bit 模型要訓練前的標準準備，包括打開 gradient checkpointing（用重算換記憶體）
3. **`PeftModel.from_pretrained(..., is_trainable=True)`**：載入 SFT adapter，**`is_trainable=True` 代表這個 adapter 要繼續訓練**。沒加的話，adapter 會被凍結，DPO 什麼都學不到
4. **把 LoRA 權重轉成 fp32**：QLoRA 的標準組合是「底模 4-bit、計算 fp16、LoRA 權重 fp32」。fp16 混合精度訓練裡，負責防止數值爆掉的 GradScaler **只處理 fp32 權重**。第二次在 Kaggle 跑時，你的 SFT adapter 載入後是 **bf16**，訓練第一步就出現 `not implemented for 'BFloat16'` 的錯誤，所以這裡統一轉成 fp32

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                       activation_checkpointing_kwargs={"use_reentrant": False})
model = PeftModel.from_pretrained(base, ADAPTER_ID, is_trainable=True)
model.config.use_cache = False      # 訓練時不需要 KV cache，關掉省記憶體

# T4 用 fp16 混合精度訓練：負責防止數值爆掉的 GradScaler 只處理 fp32 權重。
# 你的 SFT adapter 載入後是 bf16，所以把要訓練的 LoRA 權重轉成 fp32（QLoRA 的標準做法）。
cast = collections.Counter()
for n, p in model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        cast[str(p.dtype)] += 1
        p.data = p.data.float()
assert all(p.dtype == torch.float32 for p in model.parameters() if p.requires_grad), "❌ 還有可訓練權重不是 fp32"
print("LoRA 權重轉成 fp32：", dict(cast) or "原本就是 fp32")

cfg = model.peft_config["default"]
print(f"載入完成 {time.time()-t0:.0f}s | adapter r={cfg.r}, alpha={cfg.lora_alpha}, modules={sorted(cfg.target_modules)}")
model.print_trainable_parameters()
print(f"GPU 記憶體：{torch.cuda.memory_allocated()/1e9:.1f} GB")

## 8. 建立 DPOTrainer

### `DPOConfig` 還沒介紹到的參數

| 參數 | 值 | 意思 |
|---|---|---|
| `lr_scheduler_type` | cosine | 學習率先上升再慢慢降到 0，像煞車一樣，最後幾步只做微調 |
| `warmup_steps` | 0.1 | 前 10% 的步數，學習率從 0 慢慢升上去，避免一開始就調太猛。填小數代表比例，填整數代表步數（transformers 5 拿掉了舊的 `warmup_ratio`，改用這個） |
| `fp16` | True | 混合精度訓練。T4 不支援 bf16，所以用 fp16 |
| `activation_checkpointing` | True | 不保留中間結果，需要時再重算。慢約 20%，但省很多記憶體（舊名稱叫 `gradient_checkpointing`，新版改名了） |
| `optim` | paged_adamw_8bit | 8-bit 的 AdamW 優化器，省記憶體。**跟你 SFT 時一樣** |
| `max_grad_norm` | 0.5 | 梯度裁剪：每一步更新的力道超過 0.5 就壓回 0.5，避免一步跨太大。**跟你 SFT 時一樣** |
| `eval_strategy` / `eval_steps` | steps / 25 | 每 25 步考一次小考卷 |
| `save_steps` / `save_total_limit` | 50 / 1 | 每 50 步存一次，只留最新一份 |
| `report_to` 等 | trackio | 見 3.5。另外第 11 格也會自己畫一份 PNG，跟 adapter 一起上傳 |

`ref_model=None`：不另外給參考模型，讓 TRL 自動把 SFT adapter 複製成 `ref`。

In [ ]:
from trl import DPOConfig, DPOTrainer

args = DPOConfig(
    output_dir=CKPT, seed=SEED,
    beta=BETA, loss_type=LOSS_TYPE, loss_weights=LOSS_WEIGHTS,
    learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine", warmup_steps=0.1,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, max_length=MAX_LENGTH,
    fp16=True, activation_checkpointing=True, activation_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", max_grad_norm=0.5,
    logging_steps=5, eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=1,
    report_to="trackio", project=TRACKIO_PROJECT, run_name=RUN_NAME,
    trackio_space_id=TRACKIO_SPACE_ID, hub_private_repo=True, trackio_static_space_id=False,
)
trainer = DPOTrainer(model=model, ref_model=None, args=args,
                     train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tokenizer)
print("adapter 清單:", list(trainer.model.peft_config))

## 9. 訓練前的安全檢查：參考模型真的是 SFT 嗎？

這是整本 notebook **最重要的一格**。TRL 在某些情況下不會複製 adapter，而是改用「把 adapter 關掉」的底模當參考。如果發生這種事，DPO 比較的對象就會變成原始 Qwen，而不是你的 SFT 模型，整個訓練的方向就錯了。

檢查步驟，任何一道不通過就**立刻停下**，不浪費 2 小時：

1. **有沒有 `ref` adapter**
2. **診斷**：印出兩邊的精度和最大差距，並確認 ref **不是**全新的空白 adapter（空白 adapter 的 `lora_B` 全是 0）
3. **強制同步**：把 SFT adapter 的權重**連同精度**直接複製給 ref，讓兩者位元完全相同
4. **考一次小考卷**：訓練前，訓練中的模型 = 參考模型，所以 `rewards/margins` 應該是 **0**。如果參考模型是原始 Qwen，差距會是好幾個單位，不會接近 0

> **為什麼要第 3 步？** 第一次在 Kaggle 跑時，「ref 跟 SFT 位元完全相同」這個檢查沒過。第二次印出診斷才知道原因：SFT adapter 載入後是 **bf16**，TRL 新建的 ref 是 **fp32**，最大差距 4.7e-4。現在第 7 格會先把 LoRA 權重統一轉成 fp32，兩邊應該一開始就一致；這一步留著當保險。

這次小考的結果也是「訓練前」的基準線。

In [ ]:
assert "ref" in trainer.model.peft_config, "❌ TRL 沒有建立 ref adapter，參考模型會變成原始 Qwen"

params = dict(trainer.model.named_parameters())
pairs_checked = [(n, n.replace(".default.", ".ref.")) for n in params if ".default." in n]
assert pairs_checked and all(b in params for _, b in pairs_checked), "❌ 有 default 權重找不到對應的 ref 權重"

# (1) 診斷：兩邊的精度、差距、ref 是不是空白 adapter
dtypes = {(str(params[a].dtype), str(params[b].dtype)) for a, b in pairs_checked}
max_diff = max((params[a].float() - params[b].float()).abs().max().item() for a, b in pairs_checked)
ref_B_all_zero = all(params[b].abs().max().item() == 0 for a, b in pairs_checked if "lora_B" in a)
print(f"診斷：(default 精度, ref 精度) = {dtypes}；最大差距 = {max_diff:.2e}；ref 的 lora_B 全是 0 = {ref_B_all_zero}")
assert not ref_B_all_zero, "❌ ref 是全新的空白 adapter，沒有複製到 SFT 權重"

# (2) 強制同步：把 SFT adapter 的權重（連同精度）複製給 ref，兩者就完全一樣
with torch.no_grad():
    for a, b in pairs_checked:
        params[b].data = params[a].detach().clone()
        params[b].requires_grad_(False)
params = dict(trainer.model.named_parameters())
assert all(torch.equal(params[a], params[b]) for a, b in pairs_checked), "❌ 同步後 ref 還是跟 SFT adapter 不一樣"

# (3) 只有 default 會被訓練
trainable = {("default" if ".default." in n else "ref" if ".ref." in n else "other")
             for n, p in trainer.model.named_parameters() if p.requires_grad}
assert trainable == {"default"}, f"❌ 會被訓練的參數不只 default：{trainable}"
print(f"✅ ref adapter 已和 SFT adapter 完全一致（{len(pairs_checked)} 個權重），只有 default 會被訓練")

t0 = time.time()
PRE_EVAL = trainer.evaluate()
print(f"訓練前小考（{time.time()-t0:.0f}s）:", {k: round(v, 4) for k, v in PRE_EVAL.items() if "rewards" in k or "logps" in k or k == "eval_loss"})
assert abs(PRE_EVAL["eval_rewards/margins"]) < 1e-2 and abs(PRE_EVAL["eval_rewards/chosen"]) < 1e-2, "❌ 訓練前 rewards 不是 0，參考模型不是 SFT"
print("✅ 訓練前 rewards/margins ≈ 0，參考模型確實是 SFT 模型")
print(f"   SFT 模型給正確答案的平均 log 機率 {PRE_EVAL['eval_logps/chosen']:.1f}，給陷阱答案 {PRE_EVAL['eval_logps/rejected']:.1f}")

## 10. 開始訓練

### 怎麼看訓練紀錄

每 5 步印一次，每 25 步考一次小考卷。**打開 Trackio 儀表板看曲線最清楚**（網址在 3.5 那一格的輸出）。最重要的幾個指標：

| 指標 | 意思 | 希望看到 |
|---|---|---|
| `loss` | 總 loss（DPO + SFT） | 往下降 |
| `rewards/chosen` | 模型比 SFT **更**喜歡正確答案多少 | ≥ 0，至少不要一直往下掉 |
| `rewards/rejected` | 模型比 SFT **更**喜歡陷阱答案多少 | 往下降（越來越不喜歡陷阱） |
| `rewards/margins` | chosen − rejected | 往上升 |
| `rewards/accuracies` | 有幾成配對「正確答案的分數 > 陷阱答案」 | 接近 1 |
| `logps/chosen` | 正確答案的 log 機率 | **不要明顯變小**，變小就代表遇到 DPO 的陷阱：正確答案也被壓低了 |

### 預期

陷阱答案是規則造出來的、跟正確答案差異明確，所以 `accuracies` 應該很快就會接近 1。真正要看的是**小考卷**和 03 的正式考卷：模型是真的學會了，還是只是背熟練習本。

In [ ]:
t0 = time.time()
train_result = trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
print(f"✅ 訓練完成，用時 {TRAIN_SECONDS/60:.1f} 分鐘")
POST_EVAL = trainer.evaluate()
print("訓練後小考:", {k: round(v, 4) for k, v in POST_EVAL.items() if "rewards" in k or "logps" in k or k == "eval_loss"})

## 11. 畫出訓練曲線

把紀錄存成 CSV，並畫出四張圖：loss、rewards（chosen vs rejected）、margins、正確答案的 log 機率。這些圖會跟 adapter 一起上傳，寫報告和放作品集時用得到。

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hist = pd.DataFrame(trainer.state.log_history)
hist.to_csv(f"{LOG_DIR}/dpo_log_history.csv", index=False)
tr_ = hist[hist["loss"].notna()] if "loss" in hist else hist.iloc[0:0]
ev_ = hist[hist["eval_loss"].notna()] if "eval_loss" in hist else hist.iloc[0:0]

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
panels = [("loss", "eval_loss", "Loss"),
          ("rewards/margins", "eval_rewards/margins", "Reward margin (chosen - rejected)"),
          ("rewards/chosen", "eval_rewards/chosen", "Reward: chosen"),
          ("logps/chosen", "eval_logps/chosen", "Log-prob of chosen")]
for a, (k_tr, k_ev, title) in zip(ax.flat, panels):
    if k_tr in tr_: a.plot(tr_["step"], tr_[k_tr], label="train")
    if k_ev in ev_: a.plot(ev_["step"], ev_[k_ev], "o-", label="eval")
    a.set_title(title); a.set_xlabel("step"); a.legend(); a.grid(alpha=.3)
if "rewards/rejected" in tr_:
    ax[1, 0].plot(tr_["step"], tr_["rewards/rejected"], label="train rejected"); ax[1, 0].legend()
plt.tight_layout(); plt.savefig(f"{LOG_DIR}/dpo_curves.png", dpi=120)
print(f"存檔：{LOG_DIR}/dpo_curves.png")

## 12. 先把 adapter 存起來

**訓練完第一件事就是存檔。** 後面的冒煙測試或上傳萬一出錯，2 小時的訓練成果也不會不見。

**`selected_adapters=["default"]`**：模型裡現在有 `default` 和 `ref` 兩個 adapter，我們只要存訓練過的 `default`。

In [ ]:
trainer.model.save_pretrained(OUT_DIR, selected_adapters=["default"])
tokenizer.save_pretrained(OUT_DIR)
print("✅ adapter 已存檔：", sorted(os.listdir(OUT_DIR)))

## 13. 冒煙測試：讓 DPO 後的模型實際回答幾題

從小考卷裡每種陷阱各抽一題，讓模型（`default` adapter = SFT+DPO）用 greedy 回答，再用評分函式批改。

這只是確認模型**沒有被訓練壞**（例如輸出不再是合法 JSON）。沒通過只會顯示警告、不會停下，因為 adapter 已經存好了。真正的分數要看 03：三份正式考卷 + 原始 Qwen / SFT / DPO 一起比較。

In [ ]:
import json

def parse(text):
    """Turn model text into a dict, or None if it is not valid JSON."""
    t = text.strip()
    if t.startswith("```"):                      # strip ```json ... ``` fences
        t = t.strip("`")
        t = t[t.find("{"):] if "{" in t else t
    try:
        obj = json.loads(t)
    except Exception:
        s, e = t.find("{"), t.rfind("}")
        if s == -1 or e <= s:
            return None
        try:
            obj = json.loads(t[s:e + 1])
        except Exception:
            return None
    return obj if isinstance(obj, dict) else None

def _key(c):
    return (c.get("workload_id"), c.get("field"))

def _val(c):
    return {"value": json.dumps(c.get("value"), ensure_ascii=False),
            "valid_until": c.get("valid_until"),
            "action": c.get("action")}

def grade(pred_text, gold_text):
    """Compare intent + constraints (order-insensitive). explanation is not graded.

    Returns {"correct": bool, "errors": [tags], "exact_str": bool}.
    """
    gold = json.loads(gold_text)
    out = {"exact_str": pred_text.strip() == gold_text.strip()}
    pred = parse(pred_text)
    if pred is None:
        return {**out, "correct": False, "errors": ["json_invalid"]}

    g_int, p_int = gold.get("intent"), pred.get("intent")
    if p_int != g_int:
        if g_int == "refuse":
            tag = "missed_refuse"        # 危險：該拒絕卻照做
        elif p_int == "refuse":
            tag = "false_refuse"         # 不該拒絕卻拒絕
        else:
            tag = "wrong_intent"
        return {**out, "correct": False, "errors": [tag]}

    p_list = pred.get("constraints", [])
    if not isinstance(p_list, list) or any(not isinstance(c, dict) for c in p_list):
        return {**out, "correct": False, "errors": ["schema_invalid"]}

    g = {_key(c): _val(c) for c in gold.get("constraints", [])}
    p = {_key(c): _val(c) for c in p_list}
    errors = []
    if len(p) != len(p_list):
        errors.append("duplicate_constraint")
    if any(k not in p for k in g):
        errors.append("missing_constraint")
    if any(k not in g for k in p):
        errors.append("extra_constraint")
    for k in g:
        if k in p and p[k] != g[k]:
            for part in ("value", "valid_until", "action"):
                if p[k][part] != g[k][part]:
                    errors.append(f"wrong_{part}")
    errors = list(dict.fromkeys(errors))   # dedupe, keep order
    return {**out, "correct": not errors, "errors": errors}

trainer.model.eval()
trainer.model.set_adapter("default")
trainer.model.config.use_cache = True
picked, seen_src = [], set()
for ex in pairs["test"]:
    if ex["source"] not in seen_src:
        seen_src.add(ex["source"]); picked.append(ex)

SMOKE = []
for ex in picked:
    ids = tokenizer.apply_chat_template(ex["prompt"], add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = trainer.model.generate(**ids, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)
    g = grade(text, ex["chosen"][0]["content"])
    SMOKE.append({"source": ex["source"], "correct": g["correct"], "errors": g["errors"], "pred": text})
    print(f"[{'✅' if g['correct'] else '❌'}] {ex['source']:20s} {g['errors']}")
print(f"冒煙測試：{sum(s['correct'] for s in SMOKE)}/{len(SMOKE)} 題正確")
if not all(parse(s["pred"]) is not None for s in SMOKE):
    print("⚠️ 有輸出不是合法 JSON，模型可能被訓練壞了。adapter 已經存好，請把 02_summary.json 傳給 Claude 一起看")

## 14. 上傳到 Hugging Face

上傳的內容：
- `adapter_config.json`、`adapter_model.safetensors`：DPO adapter 本體，掛在原始 Qwen3-4B 上就能用
- tokenizer 和 chat template：跟 SFT 時一樣
- `training_logs/`：訓練紀錄 CSV、曲線圖、這次的設定 `run_config.json`
- `README.md`：模型卡，寫明這個 adapter 是怎麼來的

In [ ]:
RUN_CONFIG = {
    "method": "DPO (continue training the SFT LoRA adapter; reference = frozen copy of the SFT adapter)",
    "base_model": BASE_ID, "sft_adapter": ADAPTER_ID, "pairs": PAIRS_ID,
    "beta": BETA, "loss_type": LOSS_TYPE, "loss_weights": LOSS_WEIGHTS, "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "max_length": MAX_LENGTH,
    "quantization": "4-bit nf4, double quant, fp16 compute", "seed": SEED,
    "train_pairs": len(train_ds), "eval_pairs": len(eval_ds),
    "train_seconds": TRAIN_SECONDS, "versions": VERSIONS, "gpu": GPU_INFO,
    "trackio": {"project": TRACKIO_PROJECT, "run_name": RUN_NAME, "space": TRACKIO_SPACE_ID},
    "pre_eval": PRE_EVAL, "post_eval": POST_EVAL, "smoke_test": SMOKE, "sft_run_config": SFT_RUN_CONFIG,
}
json.dump(RUN_CONFIG, open(f"{LOG_DIR}/run_config.json", "w"), ensure_ascii=False, indent=2, default=str)

CARD = f"""---
base_model: {BASE_ID}
library_name: peft
tags: [dpo, trl, lora, qlora, grid-dispatch]
---
# grid-dispatch-A-instruct-qlora-dpo-v1

DPO-aligned LoRA adapter for the grid-dispatch secretary (Chinese operator notes → constraint JSON; refuses SLA-violating instructions).
It does not make dispatch decisions; outputs still need safety validation and human approval.

- Base: `{BASE_ID}` (4-bit nf4 during training)
- Starting point: SFT adapter `{ADAPTER_ID}`; reference model = frozen copy of the SFT adapter
- Preference data: `{PAIRS_ID}` ({len(train_ds)} train / {len(eval_ds)} eval pairs; built from the SFT train split only)
- DPO: beta={BETA}, loss={LOSS_TYPE} weights={LOSS_WEIGHTS}, lr={LEARNING_RATE}, epochs={EPOCHS}, effective batch={BATCH_SIZE*GRAD_ACCUM}
- Eval reward margin: {PRE_EVAL['eval_rewards/margins']:.3f} → {POST_EVAL['eval_rewards/margins']:.3f}; accuracy {POST_EVAL['eval_rewards/accuracies']:.3f}

Load: `PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained("{BASE_ID}"), "{OUT_REPO}")`.
Code: https://github.com/YOYU001/grid-dispatch-dpo
"""
open(f"{OUT_DIR}/README.md", "w", encoding="utf-8").write(CARD)
print("存檔：", sorted(os.listdir(OUT_DIR)))

PUSHED, PUSH_ERROR = False, ""
if PUSH_TO_HUB:
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(OUT_REPO, private=True, exist_ok=True)
        api.upload_folder(folder_path=OUT_DIR, repo_id=OUT_REPO, commit_message="DPO adapter v1")
        PUSHED = True
        print(f"✅ 已上傳到 https://huggingface.co/{OUT_REPO}（私人）")
    except Exception as e:
        PUSH_ERROR = repr(e)[:500]
        print("⚠️ 上傳失敗，adapter 仍在 Output：", PUSH_ERROR)

## 15. 摘要檔

精簡版的結果存成 `02_summary.json`，**下載後傳給 Claude**，用來決定 03 怎麼評估、要不要調整參數再練一次。

In [ ]:
summary = {
    "versions": VERSIONS, "gpu": GPU_INFO, "train_minutes": round(TRAIN_SECONDS / 60, 1),
    "params": {k: RUN_CONFIG[k] for k in ["beta", "loss_type", "loss_weights", "learning_rate", "epochs", "batch_size", "grad_accum"]},
    "steps": trainer.state.global_step,
    "pre_eval": {k: round(v, 4) for k, v in PRE_EVAL.items()},
    "post_eval": {k: round(v, 4) for k, v in POST_EVAL.items()},
    "eval_curve": ev_[["step"] + [c for c in ev_.columns if c.startswith("eval_rewards") or c in ("eval_loss", "eval_logps/chosen", "eval_logps/rejected")]].round(4).to_dict(orient="records"),
    "train_tail": tr_.tail(3)[[c for c in tr_.columns if c in ("step", "loss", "rewards/margins", "rewards/accuracies", "rewards/chosen", "logps/chosen")]].round(4).to_dict(orient="records"),
    "smoke_test": SMOKE, "pushed_to_hub": PUSHED, "push_error": PUSH_ERROR, "repo": OUT_REPO,
    "trackio": {"project": TRACKIO_PROJECT, "run_name": RUN_NAME, "space": TRACKIO_SPACE_ID},
}
json.dump(summary, open(f"{WORK}/02_summary.json", "w"), ensure_ascii=False, indent=2, default=str)
print("✅ 全部完成。到 Version → Output 下載 02_summary.json 傳給 Claude")